# Personal Wellbeing Time-Series Monitoring App

## StudentLife Data Pipeline Notebook

This notebook processes the Dartmouth StudentLife dataset into a holistic wellbeing time-series prototype aligned with KAUST's Wholebeing Digital Twin research direction.

The goal is to transform noisy, asynchronous smartphone-sensing and self-report data into daily domain scores that can support wellbeing monitoring, explainable risk detection, and future machine learning experiments.

## 1. Introduction and Data Schema Mapping

StudentLife contains several raw data streams collected at different frequencies. Some signals are passive and high-frequency, such as activity and phone lock logs. Other signals are sparse self-reports, such as stress, mood, and sleep EMAs.

This notebook maps those signals into five wellbeing domains:

| Wellbeing domain | StudentLife source | Prototype feature |
| --- | --- | --- |
| Physical | EMA Sleep + sensing/activity | Sleep duration, active-state proxy |
| Mental | EMA Stress + EMA Mood logs | Stress level, positive/negative mood proxy |
| Social | sensing/conversation + social/survey proxies | Conversation duration/frequency proxy |
| Occupational | education/deadlines + EMA Class survey | Deadline pressure, academic workload proxy |
| Digital | sensing/phonelock | Phone lock/unlock frequency and duration proxy |

The output of this notebook is a clean daily time-series CSV that can be used by the dashboard and by future ML models.

## 2. Setup

Set `STUDENTLIFE_ZIP` to the location of `dataset.zip`, or place the file in your Downloads folder.

Example:

```bash
export STUDENTLIFE_ZIP=~/Downloads/dataset.zip
```

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import re
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

DATASET_ZIP = Path(os.environ.get("STUDENTLIFE_ZIP", Path.home() / "Downloads" / "dataset.zip")).expanduser()
UID = "u00"  # Change this to inspect another student, for example u01 or u10.

if not DATASET_ZIP.exists():
    raise FileNotFoundError(f"StudentLife dataset not found: {DATASET_ZIP}")

print(f"Using dataset: {DATASET_ZIP}")

## 3. Step 1 - Raw Data Ingestion and Daily Alignment

The StudentLife dataset is asynchronous. Activity may be sampled every few seconds, conversations have start and end times, while EMAs are submitted irregularly.

The goal of this step is to load raw files for a single student and aggregate each signal into daily rows.

In [ ]:
UID_RE = re.compile(r"_u(\d+)\.")


def unix_to_day(value):
    """Convert Unix timestamp to YYYY-MM-DD."""
    try:
        return datetime.fromtimestamp(float(value)).date()
    except (TypeError, ValueError, OSError):
        return pd.NaT


def read_zip_text(zip_file, name):
    return zip_file.read(name).decode("utf-8", errors="replace")


def find_member(zip_file, contains, uid=UID, suffix=".json"):
    matches = [
        name for name in zip_file.namelist()
        if contains in name and name.endswith(suffix) and f"_{uid}" in name
    ]
    return matches[0] if matches else None


def load_ema_series(zip_file, folder_contains, value_function, column_name, uid=UID):
    member = find_member(zip_file, folder_contains, uid=uid, suffix=".json")
    if member is None:
        return pd.Series(dtype="float64", name=column_name)

    records = json.loads(read_zip_text(zip_file, member))
    rows = []
    for record in records:
        day = unix_to_day(record.get("resp_time"))
        value = value_function(record)
        if pd.notna(day) and value is not None:
            rows.append({"date": pd.Timestamp(day), column_name: value})

    if not rows:
        return pd.Series(dtype="float64", name=column_name)

    df = pd.DataFrame(rows).groupby("date")[column_name].mean().sort_index()
    return df


def safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

In [ ]:
with zipfile.ZipFile(DATASET_ZIP) as z:
    sleep = load_ema_series(
        z,
        "/EMA/response/Sleep/",
        lambda r: safe_float(r.get("hour")),
        "sleep_hours",
    )

    stress = load_ema_series(
        z,
        "/EMA/response/Stress/",
        lambda r: safe_float(r.get("level")),
        "stress_level",
    )

    mood = load_ema_series(
        z,
        "/EMA/response/Mood",
        lambda r: np.nanmean([
            safe_float(r.get("happy")) if safe_float(r.get("happy")) is not None else np.nan,
            4 - safe_float(r.get("sad")) if safe_float(r.get("sad")) is not None else np.nan,
        ]),
        "mood_raw",
    )

    class_hours = load_ema_series(
        z,
        "/EMA/response/Class/",
        lambda r: safe_float(r.get("hours")),
        "class_hours",
    )

raw_daily = pd.concat([sleep, stress, mood, class_hours], axis=1).sort_index()
raw_daily.head()

In [ ]:
def load_interval_daily(zip_file, folder_contains, start_col, end_col, output_col, uid=UID):
    member = find_member(zip_file, folder_contains, uid=uid, suffix=".csv")
    if member is None:
        return pd.Series(dtype="float64", name=output_col)

    df = pd.read_csv(zip_file.open(member))
    if start_col not in df.columns or end_col not in df.columns:
        return pd.Series(dtype="float64", name=output_col)

    df["date"] = pd.to_datetime(df[start_col], unit="s", errors="coerce").dt.date
    df["duration_minutes"] = (df[end_col] - df[start_col]).clip(lower=0) / 60
    daily = df.dropna(subset=["date"]).groupby("date")["duration_minutes"].sum()
    daily.index = pd.to_datetime(daily.index)
    daily.name = output_col
    return daily


def load_activity_daily(zip_file, uid=UID, sample_every=120):
    member = find_member(zip_file, "/sensing/activity/", uid=uid, suffix=".csv")
    if member is None:
        return pd.Series(dtype="float64", name="active_minutes")

    df = pd.read_csv(zip_file.open(member))
    df = df.iloc[::sample_every].copy()  # Keep pipeline lightweight while preserving daily signal.
    df["date"] = pd.to_datetime(df["timestamp"], unit="s", errors="coerce").dt.date
    activity_col = [col for col in df.columns if "activity" in col.lower()][0]
    df["is_active"] = df[activity_col].isin([1, 2]).astype(float)
    daily = df.dropna(subset=["date"]).groupby("date")["is_active"].mean() * 180
    daily.index = pd.to_datetime(daily.index)
    daily.name = "active_minutes"
    return daily


with zipfile.ZipFile(DATASET_ZIP) as z:
    active_minutes = load_activity_daily(z)
    phone_minutes = load_interval_daily(z, "/sensing/phonelock/", "start", "end", "phone_lock_minutes")
    conversation_minutes = load_interval_daily(z, "/sensing/conversation/", "start_timestamp", " end_timestamp", "conversation_minutes")

sensor_daily = pd.concat([active_minutes, phone_minutes, conversation_minutes], axis=1).sort_index()
sensor_daily.head()

In [ ]:
def load_deadlines(zip_file, uid=UID):
    member = "dataset/education/deadlines.csv"
    if member not in zip_file.namelist():
        return pd.Series(dtype="float64", name="deadline_pressure")

    df = pd.read_csv(zip_file.open(member))
    row = df[df["uid"].astype(str).str.zfill(2).eq(uid.replace("u", ""))]
    if row.empty:
        return pd.Series(dtype="float64", name="deadline_pressure")

    values = row.drop(columns=["uid"]).T
    values.index = pd.to_datetime(values.index, errors="coerce")
    series = pd.to_numeric(values.iloc[:, 0], errors="coerce")
    series.name = "deadline_pressure"
    return series.dropna()


with zipfile.ZipFile(DATASET_ZIP) as z:
    deadline_pressure = load_deadlines(z)

master_raw = pd.concat([raw_daily, sensor_daily, deadline_pressure], axis=1).sort_index()
master_raw = master_raw.resample("1D").mean()
master_raw.head(10)

## 4. Step 2 - Multi-Domain Scoring and Normalization

Each domain is scaled into a continuous score between `0.0` and `1.0`.

A higher score means better wellbeing for that domain. For example, high stress lowers the mental score, while high phone lock duration lowers the digital score.

In [ ]:
def clip01(series):
    return series.clip(lower=0, upper=1)


def scale_positive(series, low, high):
    """Map higher raw values to better scores."""
    return clip01((series - low) / (high - low))


def scale_negative(series, low, high):
    """Map higher raw values to worse scores, then invert."""
    return 1 - clip01((series - low) / (high - low))


aligned = master_raw.copy()

# Fill sparse EMA values with forward fill and median fallback for continuity.
aligned = aligned.ffill(limit=3)
aligned = aligned.fillna(aligned.median(numeric_only=True))

physical_score = 0.6 * scale_positive(aligned["sleep_hours"], 3, 8) + 0.4 * scale_positive(aligned["active_minutes"], 0, 60)
mental_score = 0.55 * scale_negative(aligned["stress_level"], 1, 5) + 0.45 * scale_positive(aligned["mood_raw"], 1, 3)
social_score = scale_positive(aligned["conversation_minutes"], aligned["conversation_minutes"].quantile(0.1), aligned["conversation_minutes"].quantile(0.9))
occupational_score = 0.55 * scale_negative(aligned["deadline_pressure"], 0, 3) + 0.45 * scale_negative(aligned["class_hours"], 0, 8)
digital_score = scale_negative(aligned["phone_lock_minutes"], aligned["phone_lock_minutes"].quantile(0.1), aligned["phone_lock_minutes"].quantile(0.9))

master = pd.DataFrame({
    "physical_score": physical_score,
    "mental_score": mental_score,
    "social_score": social_score,
    "occupational_score": occupational_score,
    "digital_score": digital_score,
}).clip(0, 1)

master["overall_wellbeing"] = master.mean(axis=1)
master.head()

## 5. Step 3 - Rule-Based Risk Engine

This deterministic rule engine looks for compounding multi-day patterns. The goal is not clinical diagnosis. The goal is to demonstrate explainable, context-aware risk detection.

The rule below uses a 3-day rolling window to capture cumulative sleep debt and academic pressure effects.

In [ ]:
def add_risk_flags(df):
    out = df.copy()
    rolling = out[["physical_score", "social_score", "occupational_score", "digital_score", "mental_score"]].rolling(3, min_periods=2).mean()

    occupational_strain = 1 - rolling["occupational_score"]
    previous_strain = occupational_strain.shift(3)

    strain_spike = occupational_strain > previous_strain * 1.30
    social_drop = rolling["social_score"] < rolling["social_score"].shift(3) * 0.80
    physical_drop = rolling["physical_score"] < rolling["physical_score"].shift(3) * 0.80
    mental_low = rolling["mental_score"] < 0.45

    out["burnout_risk"] = (strain_spike & (social_drop | physical_drop) & mental_low).fillna(False)
    return out


def explain_risk(row):
    if not row["burnout_risk"]:
        return "No burnout risk triggered by the current rule engine."
    return (
        "Risk flagged due to a multi-day compounding pattern: academic strain increased while "
        "mental wellbeing stayed low and either social connection or physical wellbeing declined."
    )


risk_df = add_risk_flags(master)
risk_df["risk_explanation"] = risk_df.apply(explain_risk, axis=1)
risk_df[["physical_score", "mental_score", "social_score", "occupational_score", "digital_score", "burnout_risk", "risk_explanation"]].tail(10)

## 6. Step 4 - Exploratory Data Analysis and Visualizations

These visualizations are designed to be clear enough for a recruiter or research reviewer to understand quickly.

In [ ]:
plot_cols = ["physical_score", "mental_score", "social_score", "occupational_score", "digital_score"]

fig, ax = plt.subplots(figsize=(14, 6))
risk_df[plot_cols].plot(ax=ax, linewidth=2)

for risk_day in risk_df.index[risk_df["burnout_risk"]]:
    ax.axvspan(risk_day, risk_day + pd.Timedelta(days=1), color="red", alpha=0.12)

ax.set_title(f"Multi-Domain Wellbeing Timeline for Student {UID}", fontsize=16, weight="bold")
ax.set_ylabel("Domain score, 0 poor to 1 optimal")
ax.set_xlabel("Date")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = risk_df[plot_cols + ["overall_wellbeing"]].corr()
sns.heatmap(corr, annot=True, cmap="BrBG", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation Matrix Between Wellbeing Domains", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

In [ ]:
risk_summary = {
    "student_id": UID,
    "days_analyzed": int(len(risk_df)),
    "risk_days": int(risk_df["burnout_risk"].sum()),
    "mean_overall_wellbeing": round(float(risk_df["overall_wellbeing"].mean()), 3),
    "lowest_domain_mean": risk_df[plot_cols].mean().idxmin(),
}
risk_summary

## 7. Step 5 - Clean CSV Export Pipeline

This cell exports the final daily-aligned multi-domain time-series dataset.

This structured time-series CSV serves as the clean data foundation required to test future Machine Learning experiments, such as time-series imputation with GAIN and causal discovery with PCMCI.

In [ ]:
export_df = risk_df.reset_index().rename(columns={"index": "date"})
export_df.insert(0, "student_id", UID)

output_path = Path("../data/processed_student_wellbeing_domains_u00.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
export_df.to_csv(output_path, index=False)

print(f"Exported {len(export_df)} rows to {output_path}")
export_df.head()

## Conclusion

This notebook demonstrates the full research workflow behind the Personal Wellbeing Time-Series Monitoring App:

1. Raw StudentLife data ingestion.
2. Daily alignment of asynchronous sensing and EMA signals.
3. Multi-domain wellbeing scoring.
4. Explainable rule-based burnout risk detection.
5. EDA and visual analysis.
6. Clean CSV export for future machine learning experiments.

This makes the project more than a dashboard: it becomes a small, reproducible research pipeline aligned with digital health, time-series modeling, smartphone sensing, and explainable AI.